# `Industrial Machine Learning on Hadoop and Spark`
## `Seminar 07: Spark Details`

### `Maks Nakhodnov`
#### `Bremen, 2025`

From this notebook, you can learn about:

* Pivot/Unpivot
* Window function
* UDF
* Accumulator/Broadcast

## `Initialization`

In [1]:
import os
import sys
import multiprocessing
from functools import partial
from typing import Iterator, Tuple

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_DRIVER_PYTHON'] = os.environ['PYSPARK_PYTHON'] = sys.executable

! rm -rf words.txt checkpoints

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import pyspark
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext

import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.accumulators import AccumulatorParam

import pyarrow.compute as pc

In [3]:
conf = (
    SparkConf()
        .set('spark.driver.memory','12g')
        .set('spark.ui.port', '4051')
        .setMaster('local[*]')
        # .setMaster('spark://localhost:7077')
)
sc = SparkContext(conf=conf)
spark = SparkSession(sc)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/27 22:17:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## `Cache vs Persist vs Checkpoint`

In [4]:
! echo "Hello, sample RDD" > text.txt
! echo "This RDD contains three lines" >> text.txt
! echo "This is the last line" >> text.txt
! echo "" >> text.txt
! echo "Just kidding, it contains five lines" >> text.txt

text_data = sc.textFile('text.txt')
text_data, text_data.collect()

(text.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0,
 ['Hello, sample RDD',
  'This RDD contains three lines',
  'This is the last line',
  '',
  'Just kidding, it contains five lines'])

In [5]:
distinct_words = (
    text_data
        .filter(lambda x: len(x))
        .flatMap(lambda x: x.split(' '))
        .distinct()
)
distinct_words.saveAsTextFile('words.txt')
print(distinct_words.toDebugString().decode())

(2) PythonRDD[9] at RDD at PythonRDD.scala:56 []
 |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[4] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[3] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 []
    |  PythonRDD[2] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 []
    |  text.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  text.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


To reuse computed values within the current session, you should use the `.cache` method, which stores the computation results of the given node in memory.

With `.cache`, you use only the default storage level:
* `MEMORY_ONLY` for RDD
* `MEMORY_AND_DISK` for Dataset

The `.persist` method allows you to store intermediate computations within the current session with finer control over the storage location (hard disk, RAM, etc.).

[StorageLevel](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.StorageLevel.html)

[Which Storage Level to Choose?
](https://spark.apache.org/docs/latest/rdd-programming-guide.html#which-storage-level-to-choose)

In [6]:
distinct_words_cached = distinct_words.cache()
print(distinct_words_cached.toDebugString().decode())

(2) PythonRDD[9] at RDD at PythonRDD.scala:56 [Memory Serialized 1x Replicated]
 |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:168 [Memory Serialized 1x Replicated]
 |  ShuffledRDD[4] at partitionBy at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 +-(2) PairwiseRDD[3] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 [Memory Serialized 1x Replicated]
    |  PythonRDD[2] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 [Memory Serialized 1x Replicated]
    |  text.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
    |  text.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]


In [7]:
distinct_words_cached.collect()
print(distinct_words_cached.toDebugString().decode())

(2) PythonRDD[9] at RDD at PythonRDD.scala:56 [Memory Serialized 1x Replicated]
 |       CachedPartitions: 2; MemorySize: 296.0 B; DiskSize: 0.0 B
 |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:168 [Memory Serialized 1x Replicated]
 |  ShuffledRDD[4] at partitionBy at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
 +-(2) PairwiseRDD[3] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 [Memory Serialized 1x Replicated]
    |  PythonRDD[2] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2223031120.py:5 [Memory Serialized 1x Replicated]
    |  text.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]
    |  text.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 [Memory Serialized 1x Replicated]


To save data between sessions, you can use `.checkpoint`. **The key feature of this method is that it modifies the computation graph — the computation chain for the saved RDD is removed.** Shortening the computation chain is useful in cases of large graphs, for example, in iterative algorithms.

In [8]:
distinct_first_words = (
    text_data
        .filter(lambda x: len(x))
        .flatMap(lambda x: x.split(' ')[0])
        .distinct()
)

sc.setCheckpointDir('./checkpoints')

distinct_first_words.checkpoint()
print(distinct_first_words.toDebugString().decode())

(2) PythonRDD[14] at RDD at PythonRDD.scala:56 []
 |  MapPartitionsRDD[13] at mapPartitions at PythonRDD.scala:168 []
 |  ShuffledRDD[12] at partitionBy at NativeMethodAccessorImpl.java:0 []
 +-(2) PairwiseRDD[11] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2415635507.py:5 []
    |  PythonRDD[10] at distinct at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/2415635507.py:5 []
    |  text.txt MapPartitionsRDD[1] at textFile at NativeMethodAccessorImpl.java:0 []
    |  text.txt HadoopRDD[0] at textFile at NativeMethodAccessorImpl.java:0 []


In [9]:
distinct_first_words.collect()
print(distinct_first_words.toDebugString().decode())

(2) PythonRDD[14] at RDD at PythonRDD.scala:56 []
 |  ReliableCheckpointRDD[15] at collect at /var/folders/3c/vr2463p11lz5fr_80mrg72gh0000gq/T/ipykernel_52452/656455338.py:1 []


## `Pivot`

In [10]:
! wget https://files.grouplens.org/datasets/movielens/ml-1m.zip
! unzip -o ml-1m.zip

--2025-10-27 22:17:45--  https://files.grouplens.org/datasets/movielens/ml-1m.zip
Распознаётся files.grouplens.org (files.grouplens.org)… 128.101.96.204
Подключение к files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... соединение установлено.
HTTP-запрос отправлен. Ожидание ответа… 200 OK
Длина: 5917549 (5,6M) [application/zip]
Сохранение в: «ml-1m.zip»

ml-1m.zip           100%[===================>]   5,64M  2,81MB/s    за 2,0s    

2025-10-27 22:17:48 (2,81 MB/s) - «ml-1m.zip» сохранён [5917549/5917549]

Archive:  ml-1m.zip
   creating: /Users/nakhodnov/CUB/Hadoop/Industrial-Machine-Learning-on-Hadoop-and-Spark/Seminars/Seminar 07/ml-1m
  inflating: ml-1m/movies.dat        
  inflating: ml-1m/ratings.dat       
  inflating: ml-1m/README            
  inflating: ml-1m/users.dat         


In [11]:
schema = (
    T.StructType()
        .add('movie_id', T.IntegerType())
        .add('movie', T.StringType())
        .add('categories', T.StringType())
)
movies_df = (
    spark.read.format('csv')
      .option("header", False)
      .option("sep", '::')
      .schema(schema)
      .load('./ml-1m/movies.dat')
)

schema = (
    T.StructType()
        .add('user_id', T.IntegerType())
        .add('movie_id', T.IntegerType())
        .add('rating', T.FloatType())
        .add('timestamp', T.StringType())
)
ratings_df = (
    spark.read.format('csv')
      .option("header", False)
      .option("sep", '::')
      .schema(schema)
      .load('./ml-1m/ratings.dat')
)

schema = (
    T.StructType()
        .add('user_id', T.IntegerType())
        .add('gender', T.StringType())
        .add('age', T.IntegerType())
        .add('occupation', T.IntegerType())
        .add('zip-code', T.StringType())
)
users_df = (
    spark.read.format('csv')
      .option("header", False)
      .option("sep", '::')
      .schema(schema)
      .load('./ml-1m/users.dat')
)

```
+-------+--------+------+
|user_id|movie_id|rating|
+-------+--------+------+
|      1|    1193|   5.0|
|      1|    1193|   4.0|
|      1|     661|   3.0|
|      2|     661|   3.0|
|      3|    1193|   4.0|
|      3|    2355|   5.0|
+-------+--------+------+
           
             |
             |
             V
{
    1: [(1193, 5.0), (1193, 4.0), (661, 3.0)],
    2: [(661, 3.0)],
    3: [(1193, 4.0), (2355, 5.0)]
}
           
             |
             |
             V

+-------+----+----+----+
|user_id|1193| 661|2355|
+-------+----+----+----+
|      1| 4.5| 3.0|None|
|      2|None| 3.0|None|
|      3| 4.0|None| 5.0|
+-------+----+----+----+
```

In [12]:
ratings_df.limit(10).toPandas()

,user_id,movie_id,rating,timestamp
0,1,1193,5.0,978300760
1,1,661,3.0,978302109
2,1,914,3.0,978301968
3,1,3408,4.0,978300275
4,1,2355,5.0,978824291
5,1,1197,3.0,978302268
6,1,1287,5.0,978302039
7,1,2804,5.0,978300719
8,1,594,4.0,978302268
9,1,919,4.0,978301368


In [13]:
(
    ratings_df
        # What defines rows
        .groupBy(ratings_df.user_id)
        # What defines columns
        .pivot('movie_id')
        # What defines the value in each cell
        .agg(F.mean(ratings_df.rating))
    
        .limit(10)
        .toPandas()
)

25/10/27 22:17:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
25/10/27 22:17:58 WARN DAGScheduler: Broadcasting large task binary with size 2.6 MiB


,user_id,1,2,3,4,5,6,7,8,9,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
0,148,5.0,5.0,NaN,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,463,5.0,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,496,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,5.0,NaN,NaN,5.0,NaN,NaN,NaN,NaN
3,1088,4.0,NaN,3.0,NaN,2.0,4.0,3.0,3.0,1.0,...,NaN,3.0,NaN,NaN,NaN,4.0,5.0,4.0,NaN,NaN
4,471,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,1238,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1591,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,1645,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,1829,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,1580,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


You can limit the columns in the result using an additional argument:

In [14]:
top_movies_df = (
    ratings_df
        .groupBy(ratings_df.movie_id)
        .agg(
            F.count(ratings_df.rating).alias('rates'),
            F.mean(ratings_df.rating).alias('avg_rating')
        )
        .sort('rates', ascending=False)
        .limit(10)
)
    
top_movies_df.show()

+--------+-----+------------------+
|movie_id|rates|        avg_rating|
+--------+-----+------------------+
|    2858| 3428|4.3173862310385065|
|     260| 2991| 4.453694416583082|
|    1196| 2990| 4.292976588628763|
|    1210| 2883| 4.022892819979188|
|     480| 2672|3.7638473053892216|
|    2028| 2653| 4.337353938937053|
|     589| 2649| 4.058512646281616|
|    2571| 2590| 4.315830115830116|
|    1270| 2583|3.9903213317847466|
|     593| 2578|4.3518231186966645|
+--------+-----+------------------+



In [15]:
top_movies = top_movies_df.rdd.map(lambda x: x.movie_id).collect()
print(top_movies)

[2858, 260, 1196, 1210, 480, 2028, 589, 2571, 1270, 593]


In [16]:
(
    ratings_df
        .groupBy(ratings_df.user_id)
        .pivot('movie_id', top_movies)
        .agg(F.first(ratings_df.rating))
    
        .limit(10)
        .toPandas()
)

,user_id,2858,260,1196,1210,480,2028,589,2571,1270,593
0,148,NaN,5.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0
1,463,5.0,NaN,NaN,5.0,NaN,5.0,NaN,NaN,4.0,NaN
2,496,4.0,5.0,5.0,NaN,5.0,5.0,4.0,5.0,4.0,5.0
3,1088,5.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0
4,471,NaN,3.0,NaN,4.0,4.0,5.0,5.0,NaN,3.0,5.0
5,1238,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,1591,5.0,NaN,5.0,5.0,5.0,NaN,5.0,NaN,5.0,5.0
7,1645,NaN,5.0,5.0,NaN,3.0,NaN,2.0,NaN,5.0,NaN
8,1829,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0
9,1580,NaN,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,NaN


In [17]:
(
    ratings_df
        .groupBy(ratings_df.user_id)
        .pivot('movie_id', top_movies)
        .agg(F.first(ratings_df.rating))
        .fillna(3.0)
    
        .limit(10)
        .toPandas()
)

,user_id,2858,260,1196,1210,480,2028,589,2571,1270,593
0,148,3.0,5.0,4.0,5.0,5.0,5.0,5.0,5.0,5.0,4.0
1,463,5.0,3.0,3.0,5.0,3.0,5.0,3.0,3.0,4.0,3.0
2,496,4.0,5.0,5.0,3.0,5.0,5.0,4.0,5.0,4.0,5.0
3,1088,5.0,5.0,5.0,5.0,4.0,4.0,4.0,4.0,4.0,4.0
4,471,3.0,3.0,3.0,4.0,4.0,5.0,5.0,3.0,3.0,5.0
5,1238,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
6,1591,5.0,3.0,5.0,5.0,5.0,3.0,5.0,3.0,5.0,5.0
7,1645,3.0,5.0,5.0,3.0,3.0,3.0,2.0,3.0,5.0,3.0
8,1829,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,4.0
9,1580,3.0,3.0,3.0,3.0,4.0,3.0,3.0,3.0,3.0,3.0


## `Unpivot`

In [18]:
path = "./m5-forecasting-accuracy/"

! kaggle competitions download -c m5-forecasting-accuracy

import zipfile
with zipfile.ZipFile('./m5-forecasting-accuracy.zip', 'r') as zip_ref:
    zip_ref.extractall(path)

  0%|                                               | 0.00/45.8M [00:00<?, ?B/s]
100%|██████████████████████████████████████| 45.8M/45.8M [00:00<00:00, 1.68GB/s]


In [19]:
df_validation = (
    spark.read.format('csv')
      .option("inferSchema", True)
      .option("header", True)
      .option("sep", ',')
      .load(f"{path}/sales_train_validation.csv")
)
df_validation.limit(10).toPandas()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,0,1,0,0,0,2,0,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,1,0,1,0,0,1,1
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,12,15,0,0,...,0,0,1,37,3,4,6,3,2,1
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,2,0,7,3,...,0,0,1,1,6,0,0,0,0,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,0,...,1,0,0,0,0,0,0,2,0,2


### `Spark < 3.4.0`

In [20]:
unpivot_expression = "stack(2, 'd_1', d_1, 'd_2', d_2) as (d, sales)"
unpivot_df = (
    df_validation
        .select('id', F.expr(unpivot_expression))
        .where('sales > 0')
)

unpivot_df.show(truncate=False)

+-----------------------------+---+-----+
|id                           |d  |sales|
+-----------------------------+---+-----+
|HOBBIES_1_008_CA_1_validation|d_1|12   |
|HOBBIES_1_008_CA_1_validation|d_2|15   |
|HOBBIES_1_009_CA_1_validation|d_1|2    |
|HOBBIES_1_012_CA_1_validation|d_2|2    |
|HOBBIES_1_015_CA_1_validation|d_1|4    |
|HOBBIES_1_016_CA_1_validation|d_1|5    |
|HOBBIES_1_016_CA_1_validation|d_2|1    |
|HOBBIES_1_022_CA_1_validation|d_1|2    |
|HOBBIES_1_022_CA_1_validation|d_2|1    |
|HOBBIES_1_023_CA_1_validation|d_1|2    |
|HOBBIES_1_023_CA_1_validation|d_2|1    |
|HOBBIES_1_029_CA_1_validation|d_1|2    |
|HOBBIES_1_032_CA_1_validation|d_1|9    |
|HOBBIES_1_036_CA_1_validation|d_1|2    |
|HOBBIES_1_036_CA_1_validation|d_2|1    |
|HOBBIES_1_038_CA_1_validation|d_2|1    |
|HOBBIES_1_044_CA_1_validation|d_1|3    |
|HOBBIES_1_044_CA_1_validation|d_2|3    |
|HOBBIES_1_047_CA_1_validation|d_1|1    |
|HOBBIES_1_047_CA_1_validation|d_2|2    |
+-----------------------------+---

### `Spark >= 3.4.0`

In [21]:
unpivot_df = (
    df_validation
        .unpivot(
            ids=df_validation.id,
            values=[df_validation.d_1, df_validation.d_2],
            variableColumnName='d',
            valueColumnName='sales'
        )
        .where('sales > 0')
)
unpivot_df.show(truncate=False)

+-----------------------------+---+-----+
|id                           |d  |sales|
+-----------------------------+---+-----+
|HOBBIES_1_008_CA_1_validation|d_1|12   |
|HOBBIES_1_008_CA_1_validation|d_2|15   |
|HOBBIES_1_009_CA_1_validation|d_1|2    |
|HOBBIES_1_012_CA_1_validation|d_2|2    |
|HOBBIES_1_015_CA_1_validation|d_1|4    |
|HOBBIES_1_016_CA_1_validation|d_1|5    |
|HOBBIES_1_016_CA_1_validation|d_2|1    |
|HOBBIES_1_022_CA_1_validation|d_1|2    |
|HOBBIES_1_022_CA_1_validation|d_2|1    |
|HOBBIES_1_023_CA_1_validation|d_1|2    |
|HOBBIES_1_023_CA_1_validation|d_2|1    |
|HOBBIES_1_029_CA_1_validation|d_1|2    |
|HOBBIES_1_032_CA_1_validation|d_1|9    |
|HOBBIES_1_036_CA_1_validation|d_1|2    |
|HOBBIES_1_036_CA_1_validation|d_2|1    |
|HOBBIES_1_038_CA_1_validation|d_2|1    |
|HOBBIES_1_044_CA_1_validation|d_1|3    |
|HOBBIES_1_044_CA_1_validation|d_2|3    |
|HOBBIES_1_047_CA_1_validation|d_1|1    |
|HOBBIES_1_047_CA_1_validation|d_2|2    |
+-----------------------------+---

We can perform the back operation:

In [22]:
(
    unpivot_df
        .groupBy(unpivot_df.id)
        .pivot('d')
        .agg(F.sum(unpivot_df.sales))
).show(truncate=False)

+-------------------------------+----+----+
|id                             |d_1 |d_2 |
+-------------------------------+----+----+
|FOODS_1_011_CA_1_validation    |2   |1   |
|HOBBIES_1_273_CA_1_validation  |1   |NULL|
|FOODS_2_322_CA_1_validation    |NULL|1   |
|FOODS_2_011_CA_1_validation    |1   |1   |
|FOODS_3_808_CA_1_validation    |22  |18  |
|HOUSEHOLD_1_179_CA_2_validation|9   |5   |
|FOODS_1_206_CA_2_validation    |3   |NULL|
|FOODS_3_644_CA_1_validation    |3   |1   |
|FOODS_1_054_CA_2_validation    |4   |6   |
|HOUSEHOLD_1_184_CA_2_validation|5   |2   |
|FOODS_3_458_CA_1_validation    |10  |5   |
|HOUSEHOLD_1_319_CA_3_validation|1   |1   |
|HOBBIES_1_320_CA_3_validation  |7   |2   |
|HOUSEHOLD_2_140_CA_3_validation|NULL|1   |
|FOODS_3_184_CA_2_validation    |2   |4   |
|FOODS_2_225_CA_3_validation    |4   |6   |
|FOODS_3_035_CA_3_validation    |3   |2   |
|HOBBIES_1_073_CA_4_validation  |NULL|2   |
|FOODS_3_257_CA_3_validation    |1   |NULL|
|FOODS_3_232_CA_4_validation    

## `Window function`

In [23]:
data = [
    ('James',   'Sales',     3000),
    ('Michael', 'Sales',     4600),
    ('Robert',  'Sales',     4100),
    ('Maria',   'Finance',   3000),
    ('Scott',   'Finance',   3300),
    ('Jen',     'Finance',   3900), 
    ('Jeff',    'Marketing', 3000),
    ('Kumar',   'Marketing', 2000),
    ('Saif',    'Sales',     4100)
]
 
columns= ['emp_id', 'dept_id', 'salary']
df = spark.createDataFrame(data=data, schema=columns)
print(df.printSchema())
df.toPandas()

root
 |-- emp_id: string (nullable = true)
 |-- dept_id: string (nullable = true)
 |-- salary: long (nullable = true)

None


,emp_id,dept_id,salary
0,James,Sales,3000
1,Michael,Sales,4600
2,Robert,Sales,4100
3,Maria,Finance,3000
4,Scott,Finance,3300
5,Jen,Finance,3900
6,Jeff,Marketing,3000
7,Kumar,Marketing,2000
8,Saif,Sales,4100


In [24]:
avg_salaries_df = (
    df
        .groupBy(df.dept_id)
        .agg(F.mean(df.salary).alias('avg_salary'))
)
df.join(avg_salaries_df, on='dept_id', how='inner').toPandas()

,dept_id,emp_id,salary,avg_salary
0,Sales,James,3000,3950.0
1,Sales,Michael,4600,3950.0
2,Sales,Robert,4100,3950.0
3,Finance,Maria,3000,3400.0
4,Finance,Scott,3300,3400.0
5,Finance,Jen,3900,3400.0
6,Marketing,Jeff,3000,2500.0
7,Marketing,Kumar,2000,2500.0
8,Sales,Saif,4100,3950.0


Main parameters defining a window:

1. **Partitions** — processing occurs within these independent groups — triggers a shuffle, which can potentially be slow

2. **Ordering** — the order is determined independently within each group; by default, it is undefined

3. **Window boundaries** — defined relative to each row. By default:

   * If no order is specified — the entire group is taken
   * If an order is specified — the prefix of the group up to and including the current row is taken
   
     Very large windows can be inefficient

For each row, a window is defined, and a transformation function is applied to the values within that window

In [25]:
wspec = Window.partitionBy('dept_id')
(
    df
        .withColumn(
            'salaries_list', 
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'avg_salary', 
            F.mean(df.salary).over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,avg_salary
0,Maria,Finance,3000,"[3000, 3300, 3900]",3400.0
1,Scott,Finance,3300,"[3000, 3300, 3900]",3400.0
2,Jen,Finance,3900,"[3000, 3300, 3900]",3400.0
3,Jeff,Marketing,3000,"[3000, 2000]",2500.0
4,Kumar,Marketing,2000,"[3000, 2000]",2500.0
5,James,Sales,3000,"[3000, 4600, 4100, 4100]",3950.0
6,Michael,Sales,4600,"[3000, 4600, 4100, 4100]",3950.0
7,Robert,Sales,4100,"[3000, 4600, 4100, 4100]",3950.0
8,Saif,Sales,4100,"[3000, 4600, 4100, 4100]",3950.0


To avoid unnecessary **shuffles** when applying a partitioned window, you should perform `.repartition` or `.coalesce` beforehand

In [26]:
wspec = Window.partitionBy('dept_id').orderBy('salary')
# .cache seems to be necessary, but keep in mind that all shuffles 
#    performs implicit caching so it might be a waste of resources
df = df.repartition('dept_id').cache()
(
    df
        .withColumn(
            'salaries_list', 
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'row_number', 
            F.row_number().over(wspec)
        )
        .withColumn(
            'avg_salary', 
            F.mean(df.salary).over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,row_number,avg_salary
0,James,Sales,3000,[3000],1,3000.000000
1,Robert,Sales,4100,"[3000, 4100, 4100]",2,3733.333333
2,Saif,Sales,4100,"[3000, 4100, 4100]",3,3733.333333
3,Michael,Sales,4600,"[3000, 4100, 4100, 4600]",4,3950.000000
4,Maria,Finance,3000,[3000],1,3000.000000
5,Scott,Finance,3300,"[3000, 3300]",2,3150.000000
6,Jen,Finance,3900,"[3000, 3300, 3900]",3,3400.000000
7,Kumar,Marketing,2000,[2000],1,2000.000000
8,Jeff,Marketing,3000,"[2000, 3000]",2,2500.000000


Similarly, it might be benefitial to use `.sortWithinPartitions` when using `.orderBy` in a window specification to avoid a global sort

In [27]:
wspec = (
    Window
        .partitionBy('dept_id')
        .orderBy('salary')
        .rowsBetween(Window.currentRow, Window.unboundedFollowing)
)
df = df.sortWithinPartitions('salary').cache()
(
    df
        .withColumn(
            'salaries_list', 
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'avg_salary', 
            F.mean(df.salary).over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,avg_salary
0,James,Sales,3000,"[3000, 4100, 4100, 4600]",3950.000000
1,Robert,Sales,4100,"[4100, 4100, 4600]",4266.666667
2,Saif,Sales,4100,"[4100, 4600]",4350.000000
3,Michael,Sales,4600,[4600],4600.000000
4,Maria,Finance,3000,"[3000, 3300, 3900]",3400.000000
5,Scott,Finance,3300,"[3300, 3900]",3600.000000
6,Jen,Finance,3900,[3900],3900.000000
7,Kumar,Marketing,2000,"[2000, 3000]",2500.000000
8,Jeff,Marketing,3000,[3000],3000.000000


In [28]:
wspec = (
    Window
        .partitionBy('dept_id')
        .orderBy('salary')
        .rangeBetween(-400, 400)
)
(
    df
        .withColumn(
            'salaries_list', 
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'avg_salary', 
            F.mean(df.salary).over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,avg_salary
0,James,Sales,3000,[3000],3000.0
1,Robert,Sales,4100,"[4100, 4100]",4100.0
2,Saif,Sales,4100,"[4100, 4100]",4100.0
3,Michael,Sales,4600,[4600],4600.0
4,Maria,Finance,3000,"[3000, 3300]",3150.0
5,Scott,Finance,3300,"[3000, 3300]",3150.0
6,Jen,Finance,3900,[3900],3900.0
7,Kumar,Marketing,2000,[2000],2000.0
8,Jeff,Marketing,3000,[3000],3000.0


There are many different window functions.

In [29]:
wspec = Window.partitionBy('dept_id').orderBy('salary')
(
    df
        .withColumn(
            'salaries_list',
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'avg_salary',
            F.mean(df.salary).over(wspec)
        )
        .withColumn(
            'cume_dist',
            F.cume_dist().over(wspec)
        )
        .withColumn(
            'lag',
            F.lag(df.salary, 1).over(wspec)
        )
        .withColumn(
            'lead',
            F.lead(df.salary, 1).over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,avg_salary,cume_dist,lag,lead
0,James,Sales,3000,[3000],3000.000000,0.250000,NaN,4100.0
1,Robert,Sales,4100,"[3000, 4100, 4100]",3733.333333,0.750000,3000.0,4100.0
2,Saif,Sales,4100,"[3000, 4100, 4100]",3733.333333,0.750000,4100.0,4600.0
3,Michael,Sales,4600,"[3000, 4100, 4100, 4600]",3950.000000,1.000000,4100.0,NaN
4,Maria,Finance,3000,[3000],3000.000000,0.333333,NaN,3300.0
5,Scott,Finance,3300,"[3000, 3300]",3150.000000,0.666667,3000.0,3900.0
6,Jen,Finance,3900,"[3000, 3300, 3900]",3400.000000,1.000000,3300.0,NaN
7,Kumar,Marketing,2000,[2000],2000.000000,0.500000,NaN,3000.0
8,Jeff,Marketing,3000,"[2000, 3000]",2500.000000,1.000000,2000.0,NaN


In [30]:
(
    df      
        .withColumn(
            'salaries_list', 
            F.collect_list(df.salary).over(wspec)
        )
        .withColumn(
            'nth_value', 
            F.nth_value(df.salary, 2).over(wspec)
        )
        .withColumn(
            'ntile', 
            F.ntile(2).over(wspec)
        )
        .withColumn(
            'dense_rank', 
            F.dense_rank().over(wspec)
        )
        .withColumn(
            'percent_rank', 
            F.percent_rank().over(wspec)
        )
        .withColumn(
            'rank', 
            F.rank().over(wspec)
        )
).toPandas()

,emp_id,dept_id,salary,salaries_list,nth_value,ntile,dense_rank,percent_rank,rank
0,James,Sales,3000,[3000],NaN,1,1,0.000000,1
1,Robert,Sales,4100,"[3000, 4100, 4100]",4100.0,1,2,0.333333,2
2,Saif,Sales,4100,"[3000, 4100, 4100]",4100.0,2,2,0.333333,2
3,Michael,Sales,4600,"[3000, 4100, 4100, 4600]",4100.0,2,3,1.000000,4
4,Maria,Finance,3000,[3000],NaN,1,1,0.000000,1
5,Scott,Finance,3300,"[3000, 3300]",3300.0,1,2,0.500000,2
6,Jen,Finance,3900,"[3000, 3300, 3900]",3300.0,2,3,1.000000,3
7,Kumar,Marketing,2000,[2000],NaN,1,1,0.000000,1
8,Jeff,Marketing,3000,"[2000, 3000]",3000.0,2,2,1.000000,2


## `UDF`

In [31]:
data = [
    ('James',   'Sales',     3000),
    ('Michael', 'Sales',     4600),
    ('Robert',  'Sales',     4100),
    ('Maria',   'Finance',   3000),
    ('Scott',   'Finance',   3300),
    ('Jen',     'Finance',   3900), 
    ('Jeff',    'Marketing', 3000),
    ('Kumar',   'Marketing', 2000),
    ('Saif',    'Sales',     4100),
]
 
columns= ['emp_id', 'dept_id', 'salary']
df = spark.createDataFrame(data=data, schema=columns)
df = df.repartition(3)
df.toPandas()

,emp_id,dept_id,salary
0,Robert,Sales,4100
1,Scott,Finance,3300
2,Jeff,Marketing,3000
3,Kumar,Marketing,2000
4,Saif,Sales,4100
5,James,Sales,3000
6,Michael,Sales,4600
7,Maria,Finance,3000
8,Jen,Finance,3900


### `Python UDF (Row-at-a-time)`

* Process each row individually
* Slowest (huge serialization overhead)
  - Apache Arrow can be used to speed this up
* Use only if there’s no other alternative

In [32]:
spark.sparkContext.getConf().get('spark.sql.execution.pythonUDF.arrow.enabled')

In [33]:
@F.udf(returnType=T.DoubleType(), useArrow=True)
def add_one(x):
    print(f'Call:\n{x}\n', file=sys.stdout, flush=True)
    return x + 1.0

df.withColumn('Python UDF', add_one(df.salary)).toPandas()

Call:e 89:>                                                         (0 + 3) / 3]
4100

Call:
3300
Call:
3000

Call:
4600

Call:
3000


Call:
3900

Call:
3000

Call:
2000

Call:
4100



,emp_id,dept_id,salary,Python UDF
0,Robert,Sales,4100,4101.0
1,Scott,Finance,3300,3301.0
2,Jeff,Marketing,3000,3001.0
3,Kumar,Marketing,2000,2001.0
4,Saif,Sales,4100,4101.0
5,James,Sales,3000,3001.0
6,Michael,Sales,4600,4601.0
7,Maria,Finance,3000,3001.0
8,Jen,Finance,3900,3901.0


### `Pandas UDF (Vectorized UDF)`

* Work with batches of data as Pandas Series (`spark.sql.execution.arrow.maxRecordsPerBatch`)
* Use Apache Arrow to transfer data between JVM and Python
* Do not support conditional expressions or short-circuiting in boolean expressions; everything gets executed internally. The workaround is to incorporate the condition into the functions:
```python
spark.sql("SELECT s FROM test1 WHERE s IS NOT NULL AND strlen(s) > 1")
```

#### `Series to Series`

In [34]:
spark.conf.set('spark.sql.executor.arrow.enabled', True)
spark.conf.set('spark.sql.executor.arrow.maxRecordsPerBatch', 3)

In [35]:
@F.pandas_udf(returnType=T.DoubleType())
def add_one_pandas(x: pd.Series) -> pd.Series:
    print(f'Call:\n{x}\n', file=sys.stdout, flush=True)
    return x + 1

df.withColumn('Pandas UDF', add_one_pandas(df.salary)).toPandas()

Call:
0    3000
Name: _0, dtype: int64

Call:
0    4100
1    3300
2    3000
3    2000
4    4100
Name: _0, dtype: int64

Call:
0    4600
1    3000
2    3900
Name: _0, dtype: int64



,emp_id,dept_id,salary,Pandas UDF
0,Robert,Sales,4100,4101.0
1,Scott,Finance,3300,3301.0
2,Jeff,Marketing,3000,3001.0
3,Kumar,Marketing,2000,2001.0
4,Saif,Sales,4100,4101.0
5,James,Sales,3000,3001.0
6,Michael,Sales,4600,4601.0
7,Maria,Finance,3000,3001.0
8,Jen,Finance,3900,3901.0


In [36]:
@F.pandas_udf(T.StringType())
def add_two_pandas(emp_id: pd.Series, salary: pd.Series) -> pd.Series:
    print(f'Call:\n{emp_id}\n', file=sys.stdout, flush=True)
    return emp_id + '. Salary: ' + salary.apply(str)

df.withColumn(
    'Pandas UDF. Multiple Input', add_two_pandas(df.emp_id, df.salary)
).toPandas()

Call:
0    James
Name: _0, dtype: object
Call:
0    Michael
1      Maria
2        Jen
Name: _0, dtype: object


Call:
0    Robert
1     Scott
2      Jeff
3     Kumar
4      Saif
Name: _0, dtype: object



,emp_id,dept_id,salary,Pandas UDF. Multiple Input
0,Robert,Sales,4100,Robert. Salary: 4100
1,Scott,Finance,3300,Scott. Salary: 3300
2,Jeff,Marketing,3000,Jeff. Salary: 3000
3,Kumar,Marketing,2000,Kumar. Salary: 2000
4,Saif,Sales,4100,Saif. Salary: 4100
5,James,Sales,3000,James. Salary: 3000
6,Michael,Sales,4600,Michael. Salary: 4600
7,Maria,Finance,3000,Maria. Salary: 3000
8,Jen,Finance,3900,Jen. Salary: 3900


In [37]:
schema = T.StructType([
    T.StructField('first', T.DoubleType(), nullable=True),
    T.StructField('second', T.DoubleType(), nullable=True),
])

@F.pandas_udf(schema)
def add_two_pandas(x: pd.Series) -> pd.Series:
    print(f'\nCall:\n{x}\n', file=sys.stdout, flush=True)
    return pd.DataFrame({'first': x + 1, 'second': x + 2})

df_output = df.withColumn(
    'Pandas UDF. Multiple Output', add_two_pandas(df.salary)
).cache()

display(df_output.toPandas())
df_output.select(
    F.col('`Pandas UDF. Multiple Output`').first
).toPandas()


Call:
0    4600
1    3000
2    3900
Name: _0, dtype: int64


Call:
0    4100
1    3300
2    3000
3    2000
4    4100
Name: _0, dtype: int64

Call:
0    3000
Name: _0, dtype: int64




,emp_id,dept_id,salary,Pandas UDF. Multiple Output
0,Robert,Sales,4100,"(4101.0, 4102.0)"
1,Scott,Finance,3300,"(3301.0, 3302.0)"
2,Jeff,Marketing,3000,"(3001.0, 3002.0)"
3,Kumar,Marketing,2000,"(2001.0, 2002.0)"
4,Saif,Sales,4100,"(4101.0, 4102.0)"
5,James,Sales,3000,"(3001.0, 3002.0)"
6,Michael,Sales,4600,"(4601.0, 4602.0)"
7,Maria,Finance,3000,"(3001.0, 3002.0)"
8,Jen,Finance,3900,"(3901.0, 3902.0)"


,Pandas UDF. Multiple Output.first
0,4101.0
1,3301.0
2,3001.0
3,2001.0
4,4101.0
5,3001.0
6,4601.0
7,3001.0
8,3901.0


#### `Iterator of Series to Iterator of Series`

In [38]:
@F.pandas_udf(T.IntegerType())
def add_one_iterator(
    x: Iterator[Tuple[pd.Series, pd.Series, pd.DataFrame]]
) -> Iterator[pd.Series]:
    worker_id = multiprocessing.current_process().pid
    for emp_id, salary, combined in x:
        print(f'Iteration ({worker_id}):\n{combined}\n', file=sys.stdout, flush=True)
        yield salary * (combined.salary + 1)
        

df.withColumn(
    'Pandas UDF. Iterator', add_one_iterator(
        df.emp_id, df.salary, 
        F.struct(df.emp_id, df.dept_id, df.salary)
    )
).toPandas()

Iteration (52588):
   emp_id    dept_id  salary
0  Robert      Sales    4100
1   Scott    Finance    3300
2    Jeff  Marketing    3000
3   Kumar  Marketing    2000
4    Saif      Sales    4100

Iteration (52590):
    emp_id  dept_id  salary
0  Michael    Sales    4600
1    Maria  Finance    3000
2      Jen  Finance    3900

Iteration (52589):
  emp_id dept_id  salary
0  James   Sales    3000



,emp_id,dept_id,salary,Pandas UDF. Iterator
0,Robert,Sales,4100,16814100
1,Scott,Finance,3300,10893300
2,Jeff,Marketing,3000,9003000
3,Kumar,Marketing,2000,4002000
4,Saif,Sales,4100,16814100
5,James,Sales,3000,9003000
6,Michael,Sales,4600,21164600
7,Maria,Finance,3000,9003000
8,Jen,Finance,3900,15213900


#### `Series to Scalar`

In [39]:
@F.pandas_udf(T.DoubleType())
def mean_salary(x: pd.Series) -> float:
    print(f'Call:\n{x}\n', file=sys.stdout, flush=True)
    return x.mean()

df.select(mean_salary(df.salary)).show()

+-------------------+
|mean_salary(salary)|
+-------------------+
| 3444.4444444444443|
+-------------------+



Call:
0    4100
1    3300
2    3000
3    2000
4    4100
5    3000
6    4600
7    3000
8    3900
Name: _0, dtype: int64



In [40]:
df.groupBy('dept_id').agg(mean_salary(df.salary)).toPandas()

Call:
0    3300
1    3000
2    3900
Name: _0, dtype: int64

Call:
0    3000
1    2000
Name: _0, dtype: int64

Call:
0    4100
1    4100
2    3000
3    4600
Name: _0, dtype: int64



,dept_id,mean_salary(salary)
0,Finance,3400.0
1,Marketing,2500.0
2,Sales,3950.0


### `UDTF`

In [41]:
schema = T.StructType([
    T.StructField('emp_id', T.StringType(), nullable=True),
    T.StructField('dept_id', T.StringType(), nullable=True),
    T.StructField('salary', T.IntegerType(), nullable=True),
])

@F.udtf(returnType=schema, useArrow=True)
class ProcessorUDTF:
    def eval(self, row: T.Row):
        worker_id = multiprocessing.current_process().pid
        for idx in range(row['salary'] // 1000):
            print(f'Iteration ({worker_id}):\n{idx}\n', file=sys.stdout, flush=True)
            yield row['emp_id'], row['dept_id'], idx
            
df.createOrReplaceTempView('df')
spark.udtf.register("ProcessorUDTF", ProcessorUDTF)
spark.sql("SELECT * FROM ProcessorUDTF(TABLE(SELECT * FROM df))").limit(10).show()

+------+---------+------+
|emp_id|  dept_id|salary|
+------+---------+------+
|Robert|    Sales|     0|
|Robert|    Sales|     1|
|Robert|    Sales|     2|
|Robert|    Sales|     3|
| Scott|  Finance|     0|
| Scott|  Finance|     1|
| Scott|  Finance|     2|
|  Jeff|Marketing|     0|
|  Jeff|Marketing|     1|
|  Jeff|Marketing|     2|
+------+---------+------+



Iteration (52589):
0

Iteration (52589):
1

Iteration (52589):
2

Iteration (52589):
3

Iteration (52589):
0

Iteration (52589):
1

Iteration (52589):
2

Iteration (52589):
0

Iteration (52589):
1

Iteration (52589):
2

Iteration (52589):
0

Iteration (52589):
1

Iteration (52589):
0

Iteration (52589):
1

Iteration (52589):
2

Iteration (52589):
3



### `.applyInArrow/.applyInPandas/.applyInPandasWithState/`

**This functions require a full shuffle. All the data of a group will be loaded into memory, so the user should be aware of the potential OOM risk if data is skewed and certain groups are too large to fit in memory.**

In [42]:
schema = T.StructType([
    T.StructField('emp_id', T.StringType(), nullable=True),
    T.StructField('dept_id', T.StringType(), nullable=True),
    T.StructField('salary', T.LongType(), nullable=True),
    T.StructField('salary_norm', T.DoubleType(), nullable=True),
])

def aggregate(key, table):        
    print(f'Call:\n{key, type(table)}', file=sys.stdout, flush=True)
    salary = table.column('salary')
    salary_norm = pc.divide(
        pc.subtract(salary, pc.mean(salary)), pc.stddev(salary, ddof=1)
    )
    return table.append_column('salary_norm', salary_norm)

(
    df
        .groupBy('dept_id')
        .applyInArrow(aggregate, schema=schema)
        .show()
)

+-------+---------+------+-------------------+
| emp_id|  dept_id|salary|        salary_norm|
+-------+---------+------+-------------------+
|  Scott|  Finance|  3300|-0.2182178902359924|
|  Maria|  Finance|  3000|-0.8728715609439696|
|    Jen|  Finance|  3900|  1.091089451179962|
|   Jeff|Marketing|  3000| 0.7071067811865475|
|  Kumar|Marketing|  2000|-0.7071067811865475|
| Robert|    Sales|  4100|0.22196863065014552|
|   Saif|    Sales|  4100|0.22196863065014552|
|  James|    Sales|  3000|-1.4058013274509218|
|Michael|    Sales|  4600| 0.9618640661506307|
+-------+---------+------+-------------------+



Call:
((<pyarrow.StringScalar: 'Finance'>,), <class 'pyarrow.lib.Table'>)
Call:
((<pyarrow.StringScalar: 'Marketing'>,), <class 'pyarrow.lib.Table'>)
Call:
((<pyarrow.StringScalar: 'Sales'>,), <class 'pyarrow.lib.Table'>)


In [43]:
schema = T.StructType([
    T.StructField('emp_id', T.StringType(), nullable=True),
    T.StructField('dept_id', T.StringType(), nullable=True),
    T.StructField('salary', T.LongType(), nullable=True),
    T.StructField('salary_norm', T.DoubleType(), nullable=True),
])

def aggregate(key, table):
    print(f'Call:\n{key, type(table)}', file=sys.stdout, flush=True)
    table['salary_norm'] = (table.salary - table.salary.mean()) / table.salary.std(ddof=1)
    return table

(
    df
        .groupBy('dept_id')
        .applyInPandas(aggregate, schema=schema)
        .show()
)

+-------+---------+------+-------------------+
| emp_id|  dept_id|salary|        salary_norm|
+-------+---------+------+-------------------+
|  Scott|  Finance|  3300|-0.2182178902359924|
|  Maria|  Finance|  3000|-0.8728715609439696|
|    Jen|  Finance|  3900|  1.091089451179962|
|   Jeff|Marketing|  3000| 0.7071067811865475|
|  Kumar|Marketing|  2000|-0.7071067811865475|
| Robert|    Sales|  4100|0.22196863065014552|
|   Saif|    Sales|  4100|0.22196863065014552|
|  James|    Sales|  3000|-1.4058013274509218|
|Michael|    Sales|  4600| 0.9618640661506307|
+-------+---------+------+-------------------+



Call:
(('Finance',), <class 'pandas.core.frame.DataFrame'>)
Call:
(('Marketing',), <class 'pandas.core.frame.DataFrame'>)
Call:
(('Sales',), <class 'pandas.core.frame.DataFrame'>)


## `Custom accumulators`

In [44]:
class VectorAccumulatorParam(AccumulatorParam):
    def zero(self, value):
        return [0.0] * len(value)
    
    def addInPlace(self, value_left, value_right):
        for idx in range(len(value_left)):
             value_left[idx] += value_right[idx]
        return value_left
    
vector_acc = sc.accumulator([1.0, 2.0, 3.0], VectorAccumulatorParam())
vector_acc.value

[1.0, 2.0, 3.0]

In [45]:
def vector_add(x):
    global vector_acc
    vector_acc += [x] * 3
    
rdd = sc.parallelize([1, 2, 3])
rdd.foreach(vector_add)
vector_acc.value

[7.0, 8.0, 9.0]

## `Broadcast JOIN`

A popular use case for Broadcast is joining tables when one table is "small." In this case, it may be more efficient to send a copy of the smaller table to each worker and perform the Join locally rather than performing a distributed table join.

It’s important to note that sending large tables over the network can be expensive, so the choice between a Broadcast Join and a "regular" Join depends on the specific cluster configuration.

In [46]:
# You can set a DataFrame size threshold for which the join will automatically occur via broadcasting this table
# Size is specified in bytes. In this case — 100 MB.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 104857600)

# Value -1 disables Broadcast Join
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

In [47]:
df_validation.limit(10).toPandas()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4
5,HOBBIES_1_006_CA_1_validation,HOBBIES_1_006,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,0,1,0,0,0,2,0,0
6,HOBBIES_1_007_CA_1_validation,HOBBIES_1_007,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,1,0,1,0,0,1,1
7,HOBBIES_1_008_CA_1_validation,HOBBIES_1_008,HOBBIES_1,HOBBIES,CA_1,CA,12,15,0,0,...,0,0,1,37,3,4,6,3,2,1
8,HOBBIES_1_009_CA_1_validation,HOBBIES_1_009,HOBBIES_1,HOBBIES,CA_1,CA,2,0,7,3,...,0,0,1,1,6,0,0,0,0,0
9,HOBBIES_1_010_CA_1_validation,HOBBIES_1_010,HOBBIES_1,HOBBIES,CA_1,CA,0,0,1,0,...,1,0,0,0,0,0,0,2,0,2


In [48]:
cat_id_hex =[
    ('FOODS', '0x001'),
    ('HOUSEHOLD', '0x002'),
    ('HOBBIES', '0x003')
]
small_df = spark.createDataFrame(data=cat_id_hex, schema=['cat_id', 'hex_code'])
small_df.show()

+---------+--------+
|   cat_id|hex_code|
+---------+--------+
|    FOODS|   0x001|
|HOUSEHOLD|   0x002|
|  HOBBIES|   0x003|
+---------+--------+



In [49]:
join_df = df_validation.join(
  small_df, small_df.cat_id == df_validation.cat_id
)
display(join_df.limit(1).toPandas())
join_df.explain()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913,cat_id,hex_code
0,FOODS_1_001_CA_1_validation,FOODS_1_001,FOODS_1,FOODS,CA_1,CA,3,0,0,1,...,0,4,1,1,0,1,1,0,FOODS,0x001


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- SortMergeJoin [cat_id#227447], [cat_id#237103], Inner
   :- Sort [cat_id#227447 ASC NULLS FIRST], false, 0
   :  +- Exchange hashpartitioning(cat_id#227447, 200), ENSURE_REQUIREMENTS, [plan_id=1967]
   :     +- Filter isnotnull(cat_id#227447)
   :        +- FileScan csv [id#227444,item_id#227445,dept_id#227446,cat_id#227447,store_id#227448,state_id#227449,d_1#227450,d_2#227451,d_3#227452,d_4#227453,d_5#227454,d_6#227455,d_7#227456,d_8#227457,d_9#227458,d_10#227459,d_11#227460,d_12#227461,d_13#227462,d_14#227463,d_15#227464,d_16#227465,d_17#227466,d_18#227467,d_19#227468,... 1894 more fields] Batched: false, DataFilters: [isnotnull(cat_id#227447)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/nakhodnov/CUB/Hadoop/Industrial-Machine-Learning-on-Hadoop..., PartitionFilters: [], PushedFilters: [IsNotNull(cat_id)], ReadSchema: struct<id:string,item_id:string,dept_id:string,cat_id:string,store_id:string,state_id:stri

In [50]:
broadcast_join_df = df_validation.join(
  F.broadcast(small_df), small_df.cat_id == df_validation.cat_id
)
display(broadcast_join_df.limit(1).toPandas())
broadcast_join_df.explain()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913,cat_id,hex_code
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,1,1,3,0,1,1,HOBBIES,0x003


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [cat_id#227447], [cat_id#237103], Inner, BuildRight, false
   :- Filter isnotnull(cat_id#227447)
   :  +- FileScan csv [id#227444,item_id#227445,dept_id#227446,cat_id#227447,store_id#227448,state_id#227449,d_1#227450,d_2#227451,d_3#227452,d_4#227453,d_5#227454,d_6#227455,d_7#227456,d_8#227457,d_9#227458,d_10#227459,d_11#227460,d_12#227461,d_13#227462,d_14#227463,d_15#227464,d_16#227465,d_17#227466,d_18#227467,d_19#227468,... 1894 more fields] Batched: false, DataFilters: [isnotnull(cat_id#227447)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/nakhodnov/CUB/Hadoop/Industrial-Machine-Learning-on-Hadoop..., PartitionFilters: [], PushedFilters: [IsNotNull(cat_id)], ReadSchema: struct<id:string,item_id:string,dept_id:string,cat_id:string,store_id:string,state_id:string,d_1:...
   +- BroadcastExchange HashedRelationBroadcastMode(List(input[0, string, false]),false), [plan_id=2047]
      +- Filter is

[Spark Hints](https://jaceklaskowski.gitbooks.io/mastering-spark-sql/content/spark-sql-hint-framework.html#specifying-query-hints)